In [1]:
#import the libaras
import pandas as pd
import numpy as np
import re
import nltk
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report,accuracy_score

from nltk.stem import PorterStemmer
from imblearn.over_sampling import SMOTE


nltk.download('stopwords')
from nltk.corpus import stopwords


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ZBOOK\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
#load the data
data=pd.read_csv("/content/spam_or_not_spam.csv")


FileNotFoundError: [Errno 2] No such file or directory: '/content/spam_or_not_spam.csv'

In [ ]:
#info about data
print(data.info())
print(data.shape)
data.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   email   2999 non-null   object
 1   label   3000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 47.0+ KB
None
(3000, 2)


,email,label
2717,dear partner to be first i must apologise to y...,1
421,hindus got NUMBER billion gods so losing one i...,0
87,i need to setup a vpn between a few sites from...,0
2810,這是委託由專業廣告公司代發勿直接回信無法接收 呦 好朋友啊 許多人能力沒有比你好 因為掌握...,1
66,with our telecoms partner bumblebee don t get...,0


In [ ]:
#check nulls and duplicated
emails=data['email'].unique()
print(data.duplicated().sum())
data.isnull().sum()


127


,0
email,1
label,0


In [ ]:
#handel messing value and duplicated
data=data.drop_duplicates()
data=data.dropna()
print(data.duplicated().sum())
data.isnull().sum()

0


,0
email,0
label,0


In [ ]:
data.shape

(2872, 2)

In [ ]:
data.label.value_counts()

,count
label,
0,2445
1,427


In [ ]:
stop_words = set(stopwords.words("english"))
def cleen_text(text):
  text=text.lower()
  text=re.sub('[^a-zA-Z\s]',' ',text).split()
  text = [w for w in text if w.lower() not in stop_words]
  return text

<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-2980509142.py:4: SyntaxWarning: invalid escape sequence '\s'
  text=re.sub('[^a-zA-Z\s]',' ',text).split()


In [ ]:
steming=PorterStemmer()
def PorterStemmer(text):
  text=[steming.stem(w) for w in text]
  text=' '.join(text)
  return text

In [ ]:
data['email']=data['email'].apply(lambda x :PorterStemmer(cleen_text(x)))
data['email']

,email
0,date wed number aug number number number numbe...
1,martin post tasso papadopoulo greek sculptor b...
2,man threaten explos moscow thursday august num...
3,klez viru die alreadi prolif viru ever klez co...
4,ad cream spaghetti carbonara effect pasta make...
...,...
2995,abc good morn america rank number christma toy...
2996,hyperlink hyperlink hyperlink let mortgag lend...
2997,thank shop us gift occas free gift number numb...
2998,famou ebay market e cours learn sell complet e...


In [ ]:
tfidf=TfidfVectorizer()
X = tfidf.fit_transform(data["email"])
y=data['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)


In [ ]:

print("Before SMOTE:", X_train.shape, "Label distribution:", y_train.value_counts().to_dict())
print("After SMOTE :", X_res.shape, "Label distribution:", pd.Series(y_res).value_counts().to_dict())


Before SMOTE: (2297, 24455) Label distribution: {0: 1956, 1: 341}
After SMOTE : (3912, 24455) Label distribution: {0: 1956, 1: 1956}


In [ ]:
model=MultinomialNB()
model.fit(X_res,y_res)


MultinomialNB()

In [ ]:
y_pred = model.predict(X_test)

print("\nAccuracy:", f"{accuracy_score(y_test, y_pred):.2f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))



Accuracy: 0.97

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.97      0.98       489
           1       0.85      0.99      0.91        86

    accuracy                           0.97       575
   macro avg       0.92      0.98      0.95       575
weighted avg       0.98      0.97      0.97       575



In [ ]:
#download the model
joblib.dump(model,'the model.pkl')
joblib.dump(tfidf,'Vectorizer.pkl')